In [3]:
import modules
import torch.nn as nn

In [1]:
special_tokens = [
    "<|endoftext|>",
    "<|pad|>",
    "<|image|>",
    "<|user|>",
    "<|assistant|>",
    "<|system|>",
    "<|thinking|>",
    "<|answer|>",
    "<|image_start|>",
    "<|image_end|>",
]


In [3]:
class HiRALinear(nn.Module):
    def __init__(self, base, r=512, alpha=16):
        super().__init__()
        self.W_0 = nn.Parameter(base.weight.T, requires_grad=False)

        if base.bias is not None:
            self.bias = nn.Parameter(base.bias, requires_grad=False)
        else:
            self.bias = None


        self.d_in, self.d_out = self.W_0.shape
        self.r = r
        self.alpha = alpha
        self.scale = alpha / r

        self.A = nn.Parameter(torch.randn(self.d_in, r) * 0.01, requires_grad=True)
        self.B = nn.Parameter(torch.zeros(r, self.d_out), requires_grad=True)

    def forward(self, x):
        delta = self.W_0 * (self.A @ self.B)
        ret = x @ self.W_0 + self.scale * (x @ delta)
        if self.bias is not None:
            ret = ret + self.bias
        return ret

In [4]:
class LoRALinear(nn.Module):
    def __init__(self, base, r=512, alpha=16):
        super().__init__()
        self.W_0 = nn.Parameter(base.weight.T,requires_grad=False)
        self.bias = (nn.Parameter(base.bias, requires_grad=False) if base.bias is not None else None)
        d_in, d_out = self.W_0.shape
        self.scale = alpha / r
        self.A = nn.Parameter(torch.randn(d_in, r) * 0.01)
        self.B = nn.Parameter(torch.zeros(r, d_out))
    def forward(self, x):
        out = (x @ self.W_0 + self.scale * (x @ self.A @ self.B))
        if self.bias is not None:
            out = out + self.bias
        return out

In [4]:
import torch

vocab = modules.load_with_pickle('./data/owt_train_32009.pickle')
merges = modules.load_with_pickle('./data/owt_train_32000_merges.pickle')

tk = modules.FastTokenizerOWTHighPerformance(
    vocab,
    merges,
    special_tokens
)


FastTokenizerOWTHighPerformance: using C++ bpe_fast


In [12]:
rm_special_tokens = [
    b"<|image|>",
    b"<|system|>",
    b"<|thinking|>",
    b"<|answer|>",
    b"<|image_start|>",
    b"<|image_end|>",
]


In [11]:
type(vocab)

dict

In [9]:
len(vocab)

32010

In [13]:
new_dict = {key: value for key, value in vocab.items() if value not in rm_special_tokens}
len(new_dict)


32004

In [15]:
modules.save_with_pickle(new_dict, './data/owt_train_32004.pickle')

字典已保存为pickle文件: ./data/owt_train_32004.pickle


In [6]:
tk.encode('<|assistant|>')

[32004]

In [8]:
import torch

vocab = modules.load_with_pickle('./data/owt_train_32009.pickle')
merges = modules.load_with_pickle('./data/owt_train_32000_merges.pickle')

tk = modules.FastTokenizerOWTHighPerformance(
    vocab,
    merges,
    special_tokens
)

obj = torch.load('data/sft_EvolSft_GEM_r_512_gpt2med_iter_30000.pt', map_location='cpu')

state_dict = obj["model"]

vocab_size = 32009
context_length = 1024
batch_size = 8

d_model = 1024
num_layers = 24
num_heads = 16
d_ff = 2752
rope_theta = 10000

model = modules.TransformerLM(
        vocab_size=vocab_size,
        context_length=context_length,
        d_model=d_model,
        num_layers=num_layers,
        num_heads=num_heads,
        d_ff=d_ff,
        rope_theta=rope_theta,
    )
for l in model.layers:
    l.attn.q_proj = LoRALinear(l.attn.q_proj)
    l.attn.k_proj = LoRALinear(l.attn.k_proj)
    l.attn.v_proj = LoRALinear(l.attn.v_proj)
    l.attn.o_proj = LoRALinear(l.attn.o_proj)
model.load_state_dict(state_dict)

FastTokenizerOWTHighPerformance: using C++ bpe_fast


<All keys matched successfully>

In [23]:
import json

In [24]:
with open('./data/alpaca_evol_instruct_70k.json') as file:
    dataset = json.load(file)

In [25]:
assistant_id = 32004

print(
    model.lm_head.W[:, assistant_id].norm()
)
print(
    model.token_embeddings.embedding_weights[
        assistant_id
    ].norm()
)

tensor(1.6040, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.3308, grad_fn=<LinalgVectorNormBackward0>)


In [18]:
inp = 'hello'
inp = '<|user|> ' + inp + ' <|assistant|> '
inp = tk.encode(inp)
print(inp)


[32003, 25617, 32, 32004, 32]


In [23]:
x = torch.tensor(
        [inp],
        dtype=torch.long
    )

with torch.no_grad():
    logits = model(x)[0,-1]

topv, topi = torch.topk(logits, 30)

for v,i in zip(topv, topi):
    print(
        i.item(),
        tk.decode([i.item()]),
        v.item()
    )

45 - 7.202733039855957
292  and 6.359679698944092
32002 <|image|> 5.982621669769287
32007 <|answer|> 5.95764684677124
32003 <|user|> 5.951179027557373
32005 <|system|> 5.927049160003662
32001 <|pad|> 5.9237847328186035
287  of 5.9225382804870605
32006 <|thinking|> 5.903420925140381
32008 <|image_start|> 5.897975921630859
32004 <|assistant|> 5.8893656730651855
282  to 5.322835922241211
396  ( 5.01915168762207
46 . 4.813379287719727
44 , 4.806888580322266
474  but 4.698685169219971
450  “ 4.493773460388184
682 ,” 4.263458728790283
642 .” 4.236898899078369
329 ’ 4.202383041381836
326  for 4.2023539543151855
271 is 4.075020790100098
317  on 3.928025484085083
1223  ‘ 3.9158217906951904
405 ” 3.910421371459961
58 : 3.76021146774292
493  out 3.5614686012268066
107 k 3.5446994304656982
618  sc 3.544419288635254
2495  words 3.5413718223571777


In [10]:
out = modules.generating(model=model, enc_user_prompt=inp, end_token=32000, context_len=1024)
print(out)

tensor([32008,   287,    45,    45,  4167,  4713, 32006, 32002,    45, 32003,
          292,    45,    45,   292,    45,   292,   287,    45,    45, 32002,
        32006,    46,   389,    45,    46,    45,    45,   282,    45, 32004,
           45,  4748,  2023,    45,   287, 32006, 32002, 32008, 32004, 32003,
        32002, 32006,    45, 32007, 32008, 32003, 32005, 32002, 32007,    45,
         2913, 32007,    45, 32003, 32003,    45,    45, 32003, 32008, 32006,
        32004, 32003, 32006, 32005, 32001, 32001, 32008, 32005, 32006, 32003,
        32003, 32002, 32008, 32003, 32005, 32004, 32001, 32005, 32001, 32001,
        32007, 32003, 32004, 32005, 32003, 32005, 32004, 32004, 32005, 32008,
        32007, 32004, 32003,   684,    45, 32005, 32007, 32007, 32007, 32005,
          450,  3575, 32006, 32008, 32003, 32003, 32007, 32001, 32004, 32002,
        32001, 32004, 32008, 32006, 32006, 32003, 32007, 32001, 32001, 32005,
        32001, 32005, 32007, 32006, 32003, 32001,    45, 32001, 

In [11]:
tk.decode(out)

'<|image_start|> of-- runs rough<|thinking|><|image|>-<|user|> and-- and- and of--<|image|><|thinking|>. The-.-- to-<|assistant|>- Ty black- of<|thinking|><|image|><|image_start|><|assistant|><|user|><|image|><|thinking|>-<|answer|><|image_start|><|user|><|system|><|image|><|answer|>-”.<|answer|>-<|user|><|user|>--<|user|><|image_start|><|thinking|><|assistant|><|user|><|thinking|><|system|><|pad|><|pad|><|image_start|><|system|><|thinking|><|user|><|user|><|image|><|image_start|><|user|><|system|><|assistant|><|pad|><|system|><|pad|><|pad|><|answer|><|user|><|assistant|><|system|><|user|><|system|><|assistant|><|assistant|><|system|><|image_start|><|answer|><|assistant|><|user|> could-<|system|><|answer|><|answer|><|answer|><|system|> “ budget<|thinking|><|image_start|><|user|><|user|><|answer|><|pad|><|assistant|><|image|><|pad|><|assistant|><|image_start|><|thinking|><|thinking|><|user|><|answer|><|pad|><|pad|><|system|><|pad|><|system|><|answer|><|thinking|><|user|><|pad|>-<|pad|><

In [12]:
print(model.lm_head.W.shape)

torch.Size([1024, 32009])


In [15]:
for i in range(32000,32009):
    print(
        i,
        model.token_embeddings.embedding_weights[i].norm().item()
    )
    print(tk.decode([i]))

32000 0.5063784122467041
<|endoftext|>
32001 0.5129022002220154
<|pad|>
32002 0.5115506052970886
<|image|>
32003 3.4382123947143555
<|user|>
32004 4.330836296081543
<|assistant|>
32005 0.513224184513092
<|system|>
32006 0.491769939661026
<|thinking|>
32007 0.5054996013641357
<|answer|>
32008 0.5212087631225586
<|image_start|>


In [29]:
for n,p in model.named_parameters():
    if "A" in n:
        print(
            n,
            p.abs().mean().item(),
            p.abs().max().item()
        )
        break

layers.0.attn.q_proj.A 0.056206218898296356 0.3370341360569


In [36]:
for k in state_dict:
    if "q_proj" in k:
        print(k)

layers.0.attn.q_proj.W_0
layers.0.attn.q_proj.bias
layers.0.attn.q_proj.A
layers.0.attn.q_proj.B
layers.1.attn.q_proj.W_0
layers.1.attn.q_proj.bias
layers.1.attn.q_proj.A
layers.1.attn.q_proj.B
layers.2.attn.q_proj.W_0
layers.2.attn.q_proj.bias
layers.2.attn.q_proj.A
layers.2.attn.q_proj.B
layers.3.attn.q_proj.W_0
layers.3.attn.q_proj.bias
layers.3.attn.q_proj.A
layers.3.attn.q_proj.B
layers.4.attn.q_proj.W_0
layers.4.attn.q_proj.bias
layers.4.attn.q_proj.A
layers.4.attn.q_proj.B
layers.5.attn.q_proj.W_0
layers.5.attn.q_proj.bias
layers.5.attn.q_proj.A
layers.5.attn.q_proj.B
layers.6.attn.q_proj.W_0
layers.6.attn.q_proj.bias
layers.6.attn.q_proj.A
layers.6.attn.q_proj.B
layers.7.attn.q_proj.W_0
layers.7.attn.q_proj.bias
layers.7.attn.q_proj.A
layers.7.attn.q_proj.B
layers.8.attn.q_proj.W_0
layers.8.attn.q_proj.bias
layers.8.attn.q_proj.A
layers.8.attn.q_proj.B
layers.9.attn.q_proj.W_0
layers.9.attn.q_proj.bias
layers.9.attn.q_proj.A
layers.9.attn.q_proj.B
layers.10.attn.q_proj.W_0
laye

In [31]:
user_id = tk.vocab_inv[b"<|user|>"]
assistant_id = tk.vocab_inv[b"<|assistant|>"]

emb = model.token_embeddings.embedding_weights

print(
    emb[user_id].norm(),
    emb[assistant_id].norm()
)

tensor(0.6349, grad_fn=<LinalgVectorNormBackward0>) tensor(0.6325, grad_fn=<LinalgVectorNormBackward0>)


In [32]:
hello_id = tk.encode("hello")[0]

print(
    emb[hello_id].norm()
)

tensor(10.5550, grad_fn=<LinalgVectorNormBackward0>)


In [33]:
print(
state_dict["token_embeddings.embedding_weights"].shape
)

print(
state_dict["token_embeddings.embedding_weights"][
    user_id
].norm()
)

torch.Size([32009, 1024])
tensor(0.6349)


In [34]:
print(
tk.encode("<|assistant|>")
)

print(
tk.encode("<|assistant|> ")
)

[32004]
[32004, 32]


In [16]:
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    rank = 0
    world_size = 1

    
    context_length = 1024
    batch_size = 8

    d_model = 1024
    num_layers = 24
    num_heads = 16
    d_ff = 2752
    rope_theta = 10000

    max_iters = 41000
    eval_interval = 500
    eval_iters = 50
    log_interval = 50
    checkpoint_interval = 5000

    max_learning_rate = 1e-3
    min_learning_rate = 2e-4
    warmup_iters = 2000
    cosine_cycle_iters = max_iters

    weight_decay = 0.01
    betas = (0.9, 0.95)
    eps = 1e-8
    max_grad_norm = 1.0

In [20]:
def get_batch_from_json(
    json_data,
    batch_size,
    context_length,
    device,
    tokenizer,
    for_valid=False,
    ignore_index=-666,
):
    pad_token_id = tokenizer.vocab_inv['<|pad|>'.encode("utf-8")]

    x = torch.full(
        (batch_size, context_length),
        pad_token_id,
        dtype=torch.long,
        device=device,
    )

    y = torch.full(
        (batch_size, context_length),
        ignore_index,
        dtype=torch.long,
        device=device,
    )

    valid_end = math.floor(len(json_data) * 0.1)

    if for_valid:
        data = json_data[: valid_end]
    else:
        data = json_data[valid_end: ]

    for b in range(batch_size):
        while True:
            example = random.choice(data)
            x_text_part =  "<|user|> " + example["instruction"] + " <|assistant|> "
            x_text_full = x_text_part + example["output"] + " <|endoftext|>"
            x_token_full = tokenizer.encode(x_text_full)
            if len(x_token_full) <= context_length:
                x_token_part = tokenizer.encode(x_text_part)
                break


        labels = [ignore_index] * len(x_token_full)

        for i in range(len(x_token_part), len(x_token_full)):
            labels[i] = x_token_full[i]

        x[b, : len(x_token_full)] = torch.tensor(
            x_token_full,
            dtype=torch.long,
            device=device,
        )

        y[b, : len(labels)] = torch.tensor(
            labels,
            dtype=torch.long,
            device=device,
        )

    return x, y

x, y = get_batch_from_json(
    json_data=json_data,
    batch_size=batch_size,
    context_length=context_length,
    device=device,
    tokenizer=tk
)

print(tk.decode(x[0][:50].tolist()))
print(x[0][:50])

with torch.no_grad():
    logits = model(x[:1])

pred = logits[0].argmax(-1)

for i in range(20):
    print(
        i,
        tk.decode([x[0,i].item()]),
        "->",
        tk.decode([pred[i].item()])
    )

NameError: name 'json_data' is not defined

In [22]:
import pandas as pd

In [26]:
database = pd.read_parquet('./data/dataset/1.parquet')

In [38]:
database['text'][1]

"many views with later anarchists. Many revolutionaries of the 19th century such as William Godwin (1756–1836) and Wilhelm Weitling (1808–1871) would contribute to the anarchist doctrines of the next generation but did not use anarchist or anarchism in describing themselves or their beliefs. The first political philosopher to call himself an anarchist () was Pierre-Joseph Proudhon (1809–1865), marking the formal birth of anarchism in the mid-19th century. Since the 1890s and beginning in France, libertarianism has often been used as a synonym for anarchism and its use as a synonym is still common outside the United States. Some usages of libertarianism refer to individualistic free-market philosophy only, and free-market anarchism in particular is termed libertarian anarchism. While the term libertarian has been largely synonymous with anarchism, its meaning has more recently been diluted by wider adoption from ideologically disparate groups, including both the New Left and libertarian

In [41]:
import glob, os

In [44]:
parquet_dir = "./data/dataset"
parquet_files_paths = glob.glob(os.path.join(parquet_dir, "*.parquet"))
parquet_files_paths = [f for f in parquet_files_paths if not f.endswith(".parquet.parquet")]
parquet_files_paths

['./data/dataset/458.parquet',
 './data/dataset/446.parquet',
 './data/dataset/30.parquet',
 './data/dataset/44.parquet',
 './data/dataset/299.parquet',
 './data/dataset/60.parquet',
 './data/dataset/633.parquet',
 './data/dataset/339.parquet',
 './data/dataset/939.parquet',
 './data/dataset/750.parquet',
 './data/dataset/815.parquet',
 './data/dataset/494.parquet',
 './data/dataset/406.parquet',
 './data/dataset/754.parquet',
 './data/dataset/222.parquet',
 './data/dataset/762.parquet',
 './data/dataset/399.parquet',
 './data/dataset/106.parquet',
 './data/dataset/111.parquet',
 './data/dataset/307.parquet',
 './data/dataset/46.parquet',
 './data/dataset/393.parquet',
 './data/dataset/187.parquet',
 './data/dataset/757.parquet',
 './data/dataset/636.parquet',
 './data/dataset/767.parquet',
 './data/dataset/722.parquet',
 './data/dataset/226.parquet',
 './data/dataset/555.parquet',
 './data/dataset/929.parquet',
 './data/dataset/719.parquet',
 './data/dataset/378.parquet',
 './data/dat

In [45]:
    vocab = modules.load_with_pickle("./data/owt_train_32004.pickle")
    merges = modules.load_with_pickle("./data/owt_train_32000_merges.pickle")

In [46]:
    tk = modules.FastTokenizerOWTHighPerformance(
        vocab=vocab,
        merges=merges,
        special_tokens=["<|endoftext|>"],
        cache_size=1_000_000,
        use_cpp=True,
    )

FastTokenizerOWTHighPerformance: using C++ bpe_fast


In [47]:
tk.encode("<|endoftext|>")

[32000]

In [49]:
len(sorted(parquet_files_paths))

1000